In [ ]:
import os
import sys
import json
import time
import argparse
import random
import glob
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from matplotlib.gridspec import GridSpec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from transformers import AutoTokenizer, AutoConfig
from safetensors.torch import load_file
import pyfaidx
import dotenv
from pathlib import Path
import re
from scipy.ndimage import gaussian_filter1d

# 导入自定义模块（需确保 src 目录在 Python 路径中）
from src.dataset import MultiTrackDataset, load_fasta_sequence
from src.viewer import DatasetViewer, ResultsViewer
from src.model import GenOmics, load_finetuned_model

# 设置中文字体（根据系统环境调整）
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
class MultiTrackPredictor:
    def __init__(self, fasta_path: str, base_model_path: str, sft_ckpt_path: str,
                 tokenizer_path: str, use_flash_attn: bool,
                 index_stat_path: str = None, index_stat: dict = None,
                 **kwargs):
        self.fasta = pyfaidx.Fasta(fasta_path)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        
        # 支持直接传 dict 或从 JSON 文件读取
        if index_stat is not None:
            stat_data = index_stat
        elif index_stat_path is not None:
            stat_data = json.load(open(index_stat_path, "r"))
        else:
            raise ValueError("必须提供 index_stat（dict）或 index_stat_path（文件路径）")
        
        self.model = load_finetuned_model(
            model_class = GenOmics,
            model_path = base_model_path,
            ckpt_path = sft_ckpt_path,
            use_flash_attn = use_flash_attn,
            device = "cuda:0",
            model_init_kwargs={"index_stat": stat_data, **kwargs}
        )
        print("✔️ Model loaded successfully")
        print(self.model)
        self.model.eval()

    def predict(self, chrom: str, start: int, end: int, biosample_names: list = None) -> dict:
        """从参考基因组 FASTA 中提取序列并推理。"""
        predict_sequence = load_fasta_sequence(self.fasta, chrom, start, end)
        inputs = self.tokenizer(
            predict_sequence,
            return_tensors="pt",
            padding=False,
            truncation=True,
            max_length=32768,
            add_special_tokens=False
        ).to("cuda")
        with torch.no_grad():
            start_time = time.time()
            result = self.model.predict(inputs['input_ids'],
                                        biosample_names=biosample_names)
            time_taken = time.time() - start_time
            torch.cuda.empty_cache()
        print(f"Inference time: {time_taken:.2f} s.")
        return {
            'sequence': predict_sequence,
            'position': (chrom, start, end),
            'values': result
        }

    def predict_from_fasta_file(self, chrom: str, start: int, end: int,
                                fasta_path: str, biosample_names: list = None) -> dict:
        """从单序列 FASTA 文件（如突变序列）读取并推理。"""
        fasta_obj = pyfaidx.Fasta(fasta_path)
        predict_sequence = str(fasta_obj[0][:])
        inputs = self.tokenizer(
            predict_sequence,
            return_tensors="pt",
            padding=False,
            truncation=True,
            max_length=32768,
            add_special_tokens=False
        ).to("cuda")
        with torch.no_grad():
            start_time = time.time()
            result = self.model.predict(inputs['input_ids'],
                                        biosample_names=biosample_names)
            time_taken = time.time() - start_time
            torch.cuda.empty_cache()
        print(f"Inference time: {time_taken:.2f} s.")
        return {
            'sequence': predict_sequence,
            'position': (chrom, start, end),
            'values': result
        }

In [ ]:
# ============================================================
# 序列比对 & Indel 信号对齐工具
# ============================================================

def find_mutation_position(ref_seq: str, mut_seq: str) -> int:
    """通过逐碱基比对找到变异起始位置（0-based 窗口内坐标）。
    
    返回第一个差异碱基的位置。若一序列完全包含另一序列，则返回较短序列的长度。
    """
    min_len = min(len(ref_seq), len(mut_seq))
    for i in range(min_len):
        if ref_seq[i] != mut_seq[i]:
            return i
    return min_len


def align_indel_signal(mut_values: np.ndarray, ref_len: int,
                       mut_start_in_ref: int, length_diff: int) -> np.ndarray:
    """将突变信号数组对齐到参考序列的坐标系。
    
    处理缺失 (length_diff<0) 和插入 (length_diff>0) 两种情况，
    返回与 ref 等长的对齐后数组。缺失区域用线性插值填充。
    
    Args:
        mut_values: 突变序列预测的信号数组 [mut_len]
        ref_len: 参考序列长度
        mut_start_in_ref: 变异起始位置在 ref 坐标中的 0-based 位置
        length_diff: mut_len - ref_len（正=插入，负=缺失）
    
    Returns:
        np.ndarray: 与 ref 等长的对齐信号数组 [ref_len]
    """
    if length_diff == 0:
        # SNP: 两端补齐
        aligned = np.zeros(ref_len)
        copy_len = min(len(mut_values), ref_len)
        aligned[:copy_len] = mut_values[:copy_len]
        return aligned

    aligned = np.zeros(ref_len)

    if length_diff < 0:  # 缺失 (mut 更短)
        del_len = -length_diff
        # 1) 变异前：直接复制
        pre_len = min(mut_start_in_ref, len(mut_values))
        aligned[:pre_len] = mut_values[:pre_len]

        # 2) 缺失区域：线性插值
        left_val = mut_values[pre_len] if pre_len < len(mut_values) else 0.0
        right_val = mut_values[pre_len] if pre_len < len(mut_values) else 0.0
        fill_end = min(pre_len + del_len, ref_len)
        aligned[pre_len:fill_end] = np.linspace(left_val, right_val, fill_end - pre_len)

        # 3) 变异后：偏移复制
        src_start = pre_len
        dst_start = pre_len + del_len
        copy_len = min(len(mut_values) - src_start, ref_len - dst_start)
        if copy_len > 0:
            aligned[dst_start:dst_start + copy_len] = mut_values[src_start:src_start + copy_len]

    else:  # 插入 (mut 更长)
        pre_len = min(mut_start_in_ref, ref_len)
        aligned[:pre_len] = mut_values[:pre_len]

        # 跳过插入的 length_diff 个碱基
        src_start = pre_len + length_diff
        copy_len = min(len(mut_values) - src_start, ref_len - pre_len)
        if copy_len > 0:
            aligned[pre_len:pre_len + copy_len] = mut_values[src_start:src_start + copy_len]

    return aligned


# ============================================================
# 组合图绘制函数（track + 柱状图合为一张图）
# ============================================================

def plot_track_and_bar(ref_values, mut_values, chrom,
                       window_start, window_end,
                       gene_start, gene_end, strand,
                       exons_in_window, gene_id,
                       expression_ref, expression_mut, pct_change,
                       example_name, output_dir,
                       smoothing_sigma=10, display_pad=None):
    """将 track 图与表达量柱状图绘制在同一张图上。
    
    布局:
        ┌───────────────────────────┬──────────────┐
        │     基因轨道               │  柱状图       │
        ├───────────────────────────┤  (WT vs Mut)  │
        │     RNA-seq 信号轨道       │   + 统计      │
        │   (蓝=WT, 橙=突变)        │              │
        └───────────────────────────┴──────────────┘
    
    Args:
        ref_values: 参考信号数组（对齐后）
        mut_values: 突变信号数组（对齐到 ref 坐标系）
        exons_in_window: [(exon_start, exon_end, strand, name), ...]
        expression_ref, expression_mut: 表达量标量值
        pct_change: 表达变化百分比
        display_pad: 基因区域两侧显示扩展 bp（默认自动计算）
    """
    # 自动确定显示窗口
    if display_pad is None:
        display_pad = max(2000, (gene_end - gene_start) // 2)
    display_start = max(window_start, gene_start - display_pad)
    display_end = min(window_end, gene_end + display_pad)
    display_len = display_end - display_start

    # 提取显示区间的信号
    rel_disp_start = display_start - window_start
    rel_disp_end   = display_end - window_start
    ref_disp = ref_values[rel_disp_start:rel_disp_end].copy()
    mut_disp = mut_values[rel_disp_start:rel_disp_end].copy()

    # 高斯平滑
    if smoothing_sigma > 1.0 and len(ref_disp) > 10:
        ref_disp = gaussian_filter1d(ref_disp, smoothing_sigma)
        mut_disp = gaussian_filter1d(mut_disp, smoothing_sigma)

    x = np.arange(display_start, display_end)

    # 过滤显示区间的外显子
    exons_disp = [(s, e, st, nm) for (s, e, st, nm) in exons_in_window
                  if s < display_end and e > display_start]

    # === 创建组合布局 ===
    fig = plt.figure(figsize=(15, 4.5))
    gs = GridSpec(2, 2, width_ratios=[7, 3], height_ratios=[1.2, 3],
                  figure=fig, hspace=0.06, wspace=0.25)

    # ---- 基因轨道 ----
    ax_gene = fig.add_subplot(gs[0, 0])

    # 基因区背景高亮
    gene_color = '#2c7bb6' if strand == '+' else '#ef822f'
    g_lo = max(gene_start, display_start)
    g_hi = min(gene_end, display_end)
    if g_hi > g_lo:
        ax_gene.axvspan(g_lo, g_hi, color=gene_color, alpha=0.12, zorder=0)

    # 基因主体线
    ax_gene.plot([display_start, display_end], [0.5, 0.5],
                 color='#888888', linewidth=2, zorder=1)

    # 外显子
    for es, ee, estrand, _ in exons_disp:
        color = '#2c7bb6' if estrand == '+' else '#ef822f'
        lo = max(es, display_start)
        hi = min(ee, display_end)
        if hi > lo:
            ax_gene.barh(0.5, hi - lo, left=lo, height=0.55,
                         color=color, alpha=0.85, edgecolor=color,
                         linewidth=0.5, zorder=2)

    ax_gene.set_xlim(display_start, display_end)
    ax_gene.set_ylim(0, 1)
    ax_gene.set_ylabel(chrom, fontsize=10)
    ax_gene.set_yticks([])
    ax_gene.tick_params(labelbottom=False)
    for sp in ['top', 'left', 'right']:
        ax_gene.spines[sp].set_visible(False)

    # ---- 信号轨道 ----
    ax_sig = fig.add_subplot(gs[1, 0], sharex=ax_gene)

    ax_sig.plot(x, ref_disp, color='#4874CB', linewidth=1.2,
                alpha=0.9, label='Wild-type', zorder=3)
    ax_sig.plot(x, mut_disp, color='#ef822f', linewidth=1.2,
                alpha=0.9, linestyle='--', label='Mutant', zorder=3)

    if g_hi > g_lo:
        ax_sig.axvspan(g_lo, g_hi, color=gene_color, alpha=0.06, zorder=0)

    # x 轴格式化（显示 kb 刻度）
    xticks = np.arange(display_start, display_end + 1,
                       max(1, display_len // 5 // 1000 * 1000))
    xticklabels = [f'{p/1000:.0f}' if p >= 1000 else str(p) for p in xticks]
    ax_sig.set_xticks(xticks)
    ax_sig.set_xticklabels(xticklabels, fontsize=8)
    ax_sig.set_xlabel(f'Position on {chrom} (kb)', fontsize=10)
    ax_sig.set_ylabel('RNA-seq signal', fontsize=10)
    ax_sig.legend(fontsize=9, framealpha=0.9, loc='upper right')
    ax_sig.grid(axis='y', alpha=0.2, linestyle='--')

    # ---- 柱状图 ----
    ax_bar = fig.add_subplot(gs[:, 1])

    x_pos = np.arange(2)
    width = 0.45
    bars1 = ax_bar.bar(x_pos[0], expression_ref, width,
                       color='#4874CB', alpha=0.85, edgecolor='#4874CB',
                       linewidth=1.5, label='Wild-type')
    bars2 = ax_bar.bar(x_pos[1], expression_mut, width,
                       color='#ef822f', alpha=0.85, edgecolor='#ef822f',
                       linewidth=1.5, label='Mutant')

    # 数值标签
    for bar in bars1:
        h = bar.get_height()
        ax_bar.text(bar.get_x() + bar.get_width()/2., h,
                    f'{h:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    for bar in bars2:
        h = bar.get_height()
        ax_bar.text(bar.get_x() + bar.get_width()/2., h,
                    f'{h:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax_bar.set_xticks(x_pos)
    ax_bar.set_xticklabels(['Wild-type', 'Mutant'], fontsize=11)
    ax_bar.set_ylabel('Total Expression', fontsize=11, fontweight='bold')
    ax_bar.set_title(f'{gene_id}', fontsize=11, fontweight='bold', pad=10)
    ax_bar.legend(fontsize=10, framealpha=0.95)
    ax_bar.grid(axis='y', alpha=0.2, linestyle='--')
    ax_bar.set_axisbelow(True)

    # 统计文本框
    y_max = max(expression_ref, expression_mut) * 1.25 if max(expression_ref, expression_mut) > 0 else 1.0
    ax_bar.set_ylim(0, y_max)
    stats_text = f'Change: {pct_change:+.2f}%'
    ax_bar.text(0.5, 0.95, stats_text, transform=ax_bar.transAxes,
                fontsize=12, va='top', ha='center',
                bbox=dict(boxstyle='round', facecolor='#F5F5F5', alpha=0.9,
                          edgecolor='#CCCCCC', linewidth=1.5))

    # 保存
    fig.align_ylabels([ax_gene, ax_sig])
    plot_path = os.path.join(output_dir, f"{example_name}.png")
    fig.savefig(plot_path, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f"✅ Combined plot saved to: {plot_path}")
    return plot_path


# ============================================================
# 单突变推理管线
# ============================================================

def process_one_mutation(predictor, viewer, args, df_riceNavi,
                         example_name, mut_fasta_path):
    """对一个突变示例执行完整推理管线，返回统计字典。"""
    stats = {'example': example_name, 'status': 'OK'}

    # 获取示例信息
    row = df_riceNavi[df_riceNavi['Example'] == example_name]
    if row.empty:
        print(f"⚠️ Example '{example_name}' not found in riceNavi.txt, skipping")
        return {**stats, 'status': 'SKIP'}

    gene_ID    = row['MSU_ID'].values[0]
    chrom      = str(row['Chr'].values[0])
    gene_start = int(row['start'].values[0])
    gene_end   = int(row['end'].values[0])
    strand     = row['strand'].values[0]
    window_start = int(row['window_start'].values[0]) - 1  # → 0-based
    window_end   = int(row['window_end'].values[0]) - 1
    length_diff  = int(row['length_diff'].values[0])
    regulation   = str(row['regulation'].values[0])

    print(f"\n{'='*65}")
    print(f"Processing {example_name}: {gene_ID} @ {chrom}:{gene_start}-{gene_end} ({strand})")
    print(f"  Window: {window_start}-{window_end}  |  length_diff={length_diff}  |  Exp: {regulation}")
    print(f"{'='*65}")

    # ---- 1. 推理参考序列 ----
    print("\n>>> Predicting reference sequence ...")
    ref_pred = predictor.predict(
        chrom=chrom, start=window_start, end=window_end,
        biosample_names=args.biosample_names)

    # ---- 2. 推理突变序列 ----
    print("\n>>> Predicting mutant sequence ...")
    mut_pred = predictor.predict_from_fasta_file(
        chrom=chrom, start=window_start, end=window_end,
        fasta_path=mut_fasta_path,
        biosample_names=args.biosample_names)

    # ---- 3. 检测 ref/mut 是否互换 ----
    genome_seq = load_fasta_sequence(predictor.fasta, chrom, window_start, window_end)
    if ref_pred['sequence'] != genome_seq and mut_pred['sequence'] == genome_seq:
        print("⚠️  ref/mut 检测到互换，自动修正 ...")
        ref_pred, mut_pred = mut_pred, ref_pred

    ref_seq = ref_pred['sequence']
    mut_seq = mut_pred['sequence']
    print(f"  Ref seq length: {len(ref_seq)}  |  Mut seq length: {len(mut_seq)}")
    if ref_seq == mut_seq:
        print("⚠️  WARNING: ref and mut sequences are identical!")

    # ---- 4. 提取信号数组 ----
    assay_key = list(ref_pred['values'].keys())[0]  # 如 'total_RNA-seq_+'
    ref_raw = ref_pred['values'][assay_key][args.biosample_names].float().cpu().numpy().flatten()
    mut_raw = mut_pred['values'][assay_key][args.biosample_names].float().cpu().numpy().flatten()

    # ---- 5. 处理 Indel 对齐 ----
    if length_diff != 0:
        mut_start_in_ref = find_mutation_position(ref_seq, mut_seq)
        mut_aligned = align_indel_signal(mut_raw, len(ref_raw),
                                          mut_start_in_ref, length_diff)
        print(f"  Mutation start at window-relative pos: {mut_start_in_ref}")
    else:
        mut_aligned = mut_raw[:len(ref_raw)] if len(mut_raw) >= len(ref_raw) \
                       else np.pad(mut_raw, (0, len(ref_raw) - len(mut_raw)))

    # ---- 6. 计算基因区域表达量 ----
    gene_rel_start = gene_start - window_start
    gene_rel_end   = gene_end - window_start
    expression_ref = ref_raw[gene_rel_start:gene_rel_end].sum()
    expression_mut = mut_raw[gene_rel_start:gene_rel_end + length_diff].sum()
    pct_change = ((expression_mut - expression_ref) / expression_ref * 100) \
                 if expression_ref > 0 else 0.0

    print(f"\n  Expression — Ref: {expression_ref:.4f}  |  Mut: {expression_mut:.4f}  |  Change: {pct_change:+.2f}%")

    # ---- 7. 获取基因注释（用于 track 图） ----
    exons_in_region = []
    if viewer is not None:
        _, _, exons_in_region = viewer.get_genes_in_interval2(
            chrom, window_start, window_end)

    # ---- 8. 生成组合图 ----
    if args.save_plots:
        try:
            plot_track_and_bar(
                ref_values=ref_raw,
                mut_values=mut_aligned,
                chrom=chrom,
                window_start=window_start,
                window_end=window_end,
                gene_start=gene_start,
                gene_end=gene_end,
                strand=strand,
                exons_in_window=exons_in_region,
                gene_id=gene_ID,
                expression_ref=expression_ref,
                expression_mut=expression_mut,
                pct_change=pct_change,
                example_name=example_name,
                output_dir=args.output_dir,
                smoothing_sigma=args.smoothing_sigma,
            )
        except Exception as e:
            print(f"⚠️  Plot failed: {e}")
            import traceback
            traceback.print_exc()

    # ---- 9. 返回统计 ----
    predicted_dir = 'up' if pct_change > 0 else ('down' if pct_change < 0 else 'no_change')
    agrees = ((pct_change > 0 and 'up' in regulation) or
              (pct_change < 0 and 'down' in regulation))
    stats.update({
        'gene_id': gene_ID,
        'chrom': chrom,
        'gene_start': gene_start,
        'gene_end': gene_end,
        'strand': strand,
        'window_start': window_start,
        'window_end': window_end,
        'length_diff': length_diff,
        'expected_regulation': regulation,
        'expression_ref': round(expression_ref, 4),
        'expression_mut': round(expression_mut, 4),
        'pct_change': round(pct_change, 2),
        'predicted_direction': predicted_dir,
        'agrees_with_expected': 'Y' if agrees else 'N',
    })
    return stats

## 配置参数

In [ ]:
class Args:
    pass

args = Args()
args.mut_fasta_dir = '/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/riceNavi_output/mutant_fastas/'
args.pattern = '*.alt.fa'
args.output_dir = '../response/mutant'
args.save_plots = True
args.plot_tracks = True
args.smoothing_sigma = 10
args.biosample_names = "CSQ"

# index_stat：定义模型输出头的配置
index_stat = {
    'counts': {
        'heads': ['total_RNA-seq_+'],
        'biosample_order': [args.biosample_names],
        'target_file_name': ['CSQ_P7_1.bw'],
        'nonzero_mean': [2.2876],
    }
}

# ====== 固定路径配置（请根据实际环境修改） ======
base_model_dir = "/mnt/rice/default/Workspace/xz/hf/rice_1B_stage2_8k_hf"
sft_ckpt_path = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/output/202604020731/checkpoint-23540/model.safetensors"
fasta_path = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/osa1_r7.asm.ch.fa"
ANNOTATION_PATH = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/mutant_output_260525/modified_osa1_r7.all_models.gff3"
test_data_dir = "/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/riceNavi_output"
riceNavi_csv = os.path.join(test_data_dir, "riceNavi.txt")
# =============================================

In [ ]:
# ====== 1. 读取 riceNavi 数据 ======
if not os.path.exists(riceNavi_csv):
    print(f"Error: riceNavi.txt not found at {riceNavi_csv}")
    sys.exit(1)
df_riceNavi = pd.read_csv(riceNavi_csv, sep="\t")
print(f"✅ Loaded {len(df_riceNavi)} mutation records from riceNavi.txt")

# ====== 2. 设置随机种子 ======
seed = 42
random.seed(seed)
np.random.seed(seed)
os.environ["PYTHONHASHSEED"] = str(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
print(f"✅ Random seed set to {seed}")

# ====== 3. 初始化预测器 ======
predictor = MultiTrackPredictor(
    fasta_path=fasta_path,
    base_model_path=base_model_dir,
    sft_ckpt_path=sft_ckpt_path,
    tokenizer_path=base_model_dir,
    use_flash_attn=True,
    index_stat=index_stat,
    proj_dim=1024,
    num_downsamples=4,
    bottleneck_dim=1536
)

# ====== 4. 初始化可视化器 ======
viewer = None
if args.plot_tracks:
    if not os.path.exists(ANNOTATION_PATH):
        print(f"⚠️  Annotation file not found, track plots will be skipped.")
        args.plot_tracks = False
    else:
        print("Loading annotation for track visualization ...")
        viewer = ResultsViewer(annotation_path=ANNOTATION_PATH)

# ====== 5. 创建输出目录 ======
os.makedirs(args.output_dir, exist_ok=True)
print(f"✅ Output dir: {args.output_dir}")

✅ Random seed set to 42


2026-07-27 19:50:13,997 - INFO - ⚠️ 使用 Flash Attention 2 需要 torch.float16 或 torch.bfloat16，已自动设置为 torch.bfloat16
Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in MixtralModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", dtype=torch.float16)`
2026-07-27 19:50:32,262 - INFO - ⚠️  缺失 keys: ['output_heads.total_RNA-seq_-.weight', 'output_heads.total_RNA-seq_-.bias']...


✅ Model loaded successfully.
GenOmics(
  (base): MixtralModel(
    (embed_tokens): Embedding(128, 1024, padding_idx=14)
    (layers): ModuleList(
      (0-11): 12 x MixtralDecoderLayer(
        (self_attn): MixtralAttention(
          (q_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1024, out_features=512, bias=False)
          (v_proj): Linear(in_features=1024, out_features=512, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (block_sparse_moe): MixtralSparseMoeBlock(
          (gate): Linear(in_features=1024, out_features=8, bias=False)
          (experts): ModuleList(
            (0-7): 8 x MixtralBlockSparseTop2MLP(
              (w1): Linear(in_features=1024, out_features=4096, bias=False)
              (w2): Linear(in_features=4096, out_features=1024, bias=False)
              (w3): Linear(in_features=1024, out_features=4096, bias=False)
              (act_fn): SiLUAc

In [ ]:
# ============================================================
# 批量推理：遍历所有突变 FASTA 文件
# ============================================================

mut_fasta_list = sorted(glob.glob(os.path.join(args.mut_fasta_dir, args.pattern)))
print(f"Found {len(mut_fasta_list)} mutant FASTA files\n")

# 限定测试数量（设为 None 则跑全部）
MAX_EXAMPLES = None   # 调试时设为 3

all_stats = []
for idx, mut_fasta_path in enumerate(mut_fasta_list):
    if MAX_EXAMPLES is not None and idx >= MAX_EXAMPLES:
        print(f"\n⚠️  Reached limit of {MAX_EXAMPLES} examples, stopping early.")
        break

    # 从文件名提取 example_name（如 "example_001.alt.fa" → "example_001"）
    example_name = os.path.basename(mut_fasta_path).replace('.alt.fa', '')

    stats = process_one_mutation(
        predictor, viewer, args, df_riceNavi,
        example_name, mut_fasta_path)

    all_stats.append(stats)
    print(f"\n{'─'*65}\n")

# ============================================================
# 输出汇总 CSV
# ============================================================

df_result = pd.DataFrame(all_stats)

# 排除 'status' != 'OK' 的条目（如有 SKIP）
df_ok = df_result[df_result['status'] == 'OK'].copy()

summary_csv = os.path.join(args.output_dir, "result_summary.csv")
df_ok.to_csv(summary_csv, index=False)
print(f"\n{'='*65}")
print(f"✅ Summary CSV saved to: {summary_csv}")
print(f"   Total processed: {len(df_ok)} / {len(all_stats)}")
print(f"\nColumns: {list(df_ok.columns)}")
print(f"{'='*65}")

# 打印简要统计
up_count = (df_ok['pct_change'].astype(float) > 0).sum()
down_count = (df_ok['pct_change'].astype(float) < 0).sum()
agree_count = (df_ok['agrees_with_expected'] == 'Y').sum()
print(f"\n📊 Quick summary:")
print(f"   Up-regulated:  {up_count}")
print(f"   Down-regulated: {down_count}")
print(f"   Agree with expected: {agree_count}/{len(df_ok)} ({agree_count/len(df_ok)*100:.1f}%)" if len(df_ok) > 0 else "")
display(df_ok.head())